# PyTorch Tutorial 44: Synthetic Data Generation for LLMs (The FAANG Standard)

Data quality is the **#1 factor** in LLM performance. Not model size, not training compute -- **data**.

This tutorial covers the complete synthetic data generation pipeline used at frontier labs: from Self-Instruct and Evol-Instruct to quality filtering, decontamination, and preference pair generation. All implemented from scratch.

## Learning Objectives
1. **Build a Self-Instruct pipeline** -- generate instruction-response pairs from seed examples
2. **Implement Evol-Instruct** -- evolve instructions for complexity scaling (WizardLM approach)
3. **Generate code-specific training data** -- docstring-to-code, code-to-test, bug-fix pairs
4. **Build quality filtering from scratch** -- perplexity, deduplication (MinHash), n-gram overlap
5. **Implement decontamination** -- ensure synthetic data does not leak benchmark answers (see Tutorial 43)
6. **Generate preference pairs** -- ranked completions to (chosen, rejected) format (see Tutorial 41)
7. **Design data mixing and curriculum** -- weighted sampling and easy-to-hard scheduling

**Prerequisites**: Tutorial 41 (preference data format), Tutorial 43 (decontamination concepts)

---

## 1. Vocabulary First

- **Self-Instruct**: Pipeline where an LLM generates new instructions from a small seed set, then generates responses. Wang et al. (2022).
- **Evol-Instruct**: Iteratively evolve instructions to increase complexity via depth (add constraints, reasoning) and breadth (change domain, format). WizardLM approach.
- **Rejection Sampling**: Generate K candidate responses per prompt, score them, keep only the top-N. Used in Llama 2.
- **Decontamination**: Removing training samples that overlap with benchmark test sets to prevent data leakage. Uses n-gram matching (typically 13-grams). See Tutorial 43.
- **Data Mixing**: Combining datasets from different sources with carefully tuned ratios (e.g., 40% code, 30% math, 30% general).
- **Curriculum Learning**: Ordering training data from easy to hard, improving convergence and final performance.
- **Perplexity Filtering**: Removing samples with very high or very low perplexity (gibberish or trivially repetitive).
- **MinHash**: Locality-sensitive hashing for approximate set similarity. Used for near-duplicate detection at scale.
- **N-gram Overlap**: Measuring text similarity by comparing shared n-grams between two documents.

### Method Comparison Table

| Method | Input Required | Output | Key Strength | Key Weakness |
|--------|---------------|--------|-------------|-------------|
| Self-Instruct | Seed instructions | Instruction-response pairs | Simple, scalable | Limited diversity |
| Evol-Instruct | Existing instructions | Harder instructions | Complexity control | Can produce nonsense |
| Rejection Sampling | Prompts + scorer | Filtered responses | High quality | Expensive (K forward passes) |
| Decontamination | Dataset + benchmarks | Clean dataset | Prevents cheating | May remove valid samples |
| MinHash Dedup | Dataset | Deduplicated dataset | Fast at scale | Approximate (tunable) |
| Perplexity Filter | Dataset + LM | Filtered dataset | Removes gibberish | Needs calibration |

In [ ]:
import torch
import numpy as np
import hashlib
import json
import re
import random
import collections
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional, Set
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Ready for Synthetic Data Generation!')

---

## 2. Part 1: Self-Instruct Pipeline

Self-Instruct (Wang et al., 2022) bootstraps instruction data from a small seed set.

The pipeline:
1. Start with ~175 hand-written seed instructions
2. Sample a batch of seeds
3. Prompt an LLM to generate a NEW instruction inspired by the batch
4. Generate a response for the new instruction
5. Filter for quality (length, uniqueness, diversity)
6. Add to the instruction pool and repeat

### FAANG Interview Question

**Q: "Explain Self-Instruct and its limitations."**

**A**: Self-Instruct uses a language model to generate its own training data from seed examples. The LLM sees a few seed instructions, generates new ones, then generates responses. Limitations: (1) diversity is bounded by the seed set; (2) quality degrades over iterations as errors compound; (3) no guarantee of factual correctness; (4) generated instructions cluster around certain difficulty levels. Evol-Instruct addresses complexity; rejection sampling addresses quality.

In [ ]:
class SelfInstructPipeline:
    """Self-Instruct pipeline for generating instruction-response pairs.

    In production, generate_instruction and generate_response would call
    an LLM API. Here we use template-based generation to demonstrate
    the pipeline logic without requiring a live model.
    """

    def __init__(self):
        """Initialize with hand-written seed instructions."""
        self.seed_instructions = [
            'Explain the concept of recursion in programming.',
            'Write a Python function to reverse a string.',
            'What is the difference between a list and a tuple in Python?',
            'Describe the time complexity of binary search.',
            'How does a hash table handle collisions?',
            'Explain the observer design pattern.',
            'Write a SQL query to find duplicate records.',
            'What is the CAP theorem in distributed systems?',
            'Describe how garbage collection works in Java.',
            'Explain the difference between TCP and UDP.',
            'What is a closure in JavaScript?',
            'How does backpropagation work in neural networks?',
            'Explain the concept of polymorphism.',
            'What is the purpose of an index in a database?',
            'Describe the MVC architecture pattern.',
            'How does a load balancer distribute traffic?',
            'Explain the difference between REST and GraphQL.',
            'What is memoization and when should you use it?',
            'Describe the publish-subscribe messaging pattern.',
            'How does HTTPS encryption work?',
        ]
        self.generated_pool: List[Dict[str, str]] = []
        self.topics = [
            'algorithms', 'data structures', 'databases',
            'networking', 'design patterns', 'machine learning',
            'web development', 'security', 'concurrency', 'testing',
        ]
        self.formats = [
            'Explain', 'Compare', 'Implement', 'Describe',
            'What is', 'How does', 'Why is', 'Write',
        ]

    def generate_instruction(self, seed_batch: List[str]) -> str:
        """Generate a new instruction inspired by a batch of seeds.

        TODO: In production, prompt an LLM with the seed batch
        and ask it to generate a novel, related instruction.
        """
        fmt = random.choice(self.formats)
        topic = random.choice(self.topics)
        templates = [
            f'{fmt} the role of {topic} in modern software engineering.',
            f'{fmt} a common pitfall when working with {topic}.',
            f'{fmt} how {topic} relates to system scalability.',
            f'{fmt} the tradeoffs involved in {topic} design decisions.',
            f'{fmt} best practices for {topic} in production systems.',
        ]
        return random.choice(templates)

    def generate_response(self, instruction: str) -> str:
        """Generate a response for a given instruction.

        TODO: In production, call an LLM to generate the response.
        """
        word_count = random.randint(40, 120)
        base_words = [
            'the', 'system', 'uses', 'approach', 'which', 'provides',
            'efficient', 'handling', 'of', 'data', 'through', 'a',
            'well-designed', 'architecture', 'that', 'ensures',
            'scalability', 'and', 'reliability', 'in', 'production',
            'environments', 'by', 'implementing', 'proper', 'patterns',
        ]
        return ' '.join(random.choice(base_words) for _ in range(word_count))

    def _compute_rouge_l(self, s1: str, s2: str) -> float:
        """Compute approximate ROUGE-L between two strings via LCS."""
        words1, words2 = s1.lower().split(), s2.lower().split()
        if not words1 or not words2:
            return 0.0
        m, n = len(words1), len(words2)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if words1[i - 1] == words2[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1] + 1
                else:
                    dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
        lcs_len = dp[m][n]
        return (2.0 * lcs_len) / (m + n)

    def filter_instruction(self, instruction: str) -> bool:
        """Filter for quality: length, uniqueness (ROUGE-L < 0.7)."""
        if len(instruction) < 10 or len(instruction) > 200:
            return False
        all_existing = self.seed_instructions + [
            d['instruction'] for d in self.generated_pool
        ]
        for existing in all_existing:
            if self._compute_rouge_l(instruction, existing) > 0.7:
                return False
        return True

    def run_pipeline(self, n_target: int = 50) -> List[Dict[str, str]]:
        """Run the full Self-Instruct pipeline to generate n_target pairs."""
        attempts, max_attempts = 0, n_target * 5
        while len(self.generated_pool) < n_target and attempts < max_attempts:
            attempts += 1
            seed_batch = random.sample(self.seed_instructions, k=3)
            new_instr = self.generate_instruction(seed_batch)
            if self.filter_instruction(new_instr):
                response = self.generate_response(new_instr)
                self.generated_pool.append({
                    'instruction': new_instr, 'response': response,
                })
        return self.generated_pool


# Run the pipeline
si_pipeline = SelfInstructPipeline()
generated_data = si_pipeline.run_pipeline(n_target=50)

print(f'Generated {len(generated_data)} instruction-response pairs')
print(f'Seed pool size: {len(si_pipeline.seed_instructions)}')
print(f'\nSample generated instructions:')
for i, item in enumerate(generated_data[:5]):
    print(f'  {i+1}. {item["instruction"]}')
    print(f'     Response length: {len(item["response"].split())} words')

---

## 3. Part 2: Evol-Instruct

Evol-Instruct (WizardLM) evolves instructions through two axes:
- **Depth**: Make harder (add constraints, require reasoning, add edge cases)
- **Breadth**: Change domain or format while keeping difficulty

### FAANG Interview Question

**Q: "How does Evol-Instruct improve upon Self-Instruct?"**

**A**: Self-Instruct generates instructions at roughly the same difficulty level as the seeds. Evol-Instruct explicitly controls complexity by iteratively adding constraints (depth) or changing domains (breadth). This creates a distribution of difficulties. The key insight: LLMs trained on more complex instructions generalize better to simpler ones, but not vice versa.

In [ ]:
class EvolInstructPipeline:
    """Evol-Instruct pipeline for complexity scaling.

    Evolves instructions through depth (harder) and breadth
    (different domain) transformations over multiple generations.
    """

    def __init__(self):
        """Initialize evolution templates."""
        self.depth_templates = [
            'Additionally, handle the edge case where {edge_case}.',
            'Explain your reasoning step by step.',
            'Also consider the time and space complexity.',
            'Include error handling for invalid inputs.',
            'Optimize the solution for large-scale inputs (10M+ elements).',
        ]
        self.edge_cases = [
            'the input is empty', 'there are duplicate values',
            'the input contains negative numbers',
            'the data exceeds memory limits',
            'concurrent access is required',
        ]
        self.domains = [
            'healthcare', 'finance', 'e-commerce',
            'social media', 'autonomous vehicles', 'gaming',
        ]
        self.format_changes = [
            'Provide your answer as a numbered list.',
            'Write your answer as a technical design document.',
            'Explain this to a non-technical stakeholder.',
            'Create a comparison table with pros and cons.',
        ]

    def evolve_depth(self, instruction: str) -> str:
        """Add complexity via constraints, reasoning, or edge cases."""
        edge = random.choice(self.edge_cases)
        template = random.choice(self.depth_templates)
        return f'{instruction} {template.format(edge_case=edge)}'

    def evolve_breadth(self, instruction: str) -> str:
        """Change domain or format while keeping difficulty."""
        domain = random.choice(self.domains)
        fmt = random.choice(self.format_changes)
        return f'In the context of {domain}: {instruction} {fmt}'

    def compute_complexity_score(self, instruction: str) -> float:
        """Estimate complexity via heuristic features."""
        words = instruction.split()
        score = min(len(words) / 30.0, 1.0) * 3.0
        constraint_words = [
            'edge case', 'optimize', 'handle', 'consider',
            'additionally', 'complexity', 'error', 'concurrent',
        ]
        for cw in constraint_words:
            if cw in instruction.lower():
                score += 1.0
        if 'step by step' in instruction.lower():
            score += 1.5
        if 'and' in instruction.lower():
            score += 0.5
        return min(score, 10.0)

    def evolve(self, instruction: str, n_generations: int = 3) -> List[Dict]:
        """Chain evolutions over multiple generations."""
        history = [{
            'generation': 0, 'instruction': instruction,
            'complexity': self.compute_complexity_score(instruction),
            'evolution_type': 'seed',
        }]
        current = instruction
        for gen in range(1, n_generations + 1):
            if gen % 2 == 1:
                current = self.evolve_depth(current)
                evo_type = 'depth'
            else:
                current = self.evolve_breadth(current)
                evo_type = 'breadth'
            history.append({
                'generation': gen, 'instruction': current,
                'complexity': self.compute_complexity_score(current),
                'evolution_type': evo_type,
            })
        return history


# Demonstrate Evol-Instruct
evol = EvolInstructPipeline()
seed = 'Write a function to sort a list of integers.'
evolution_history = evol.evolve(seed, n_generations=3)

print('Evol-Instruct Evolution Chain:')
print('=' * 60)
for step in evolution_history:
    instr = step['instruction']
    display = instr[:120] + '...' if len(instr) > 120 else instr
    print(f"\nGen {step['generation']} ({step['evolution_type']}) "
          f"[complexity={step['complexity']:.1f}]:")
    print(f'  {display}')

# Visualization: complexity before/after evolution
seeds_for_viz = [
    'Explain what a hash table is.',
    'Write a function to find the maximum in a list.',
    'What is the difference between a stack and a queue?',
    'Describe how HTTP works.',
    'Implement binary search.',
    'What is a REST API?',
    'Explain polymorphism.',
    'Write a SQL query to join two tables.',
    'What is recursion?',
    'Describe the MVC pattern.',
]
before_scores = [evol.compute_complexity_score(s) for s in seeds_for_viz]
after_scores = [evol.evolve(s, 3)[-1]['complexity'] for s in seeds_for_viz]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(before_scores, bins=8, alpha=0.7, color='steelblue',
             edgecolor='black', label='Before Evolution')
axes[0].hist(after_scores, bins=8, alpha=0.7, color='coral',
             edgecolor='black', label='After 3 Generations')
axes[0].set_title('Complexity Score Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Complexity Score')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for i, s in enumerate(seeds_for_viz[:5]):
    chain = evol.evolve(s, n_generations=3)
    axes[1].plot([c['generation'] for c in chain],
                 [c['complexity'] for c in chain],
                 'o-', linewidth=2, alpha=0.7, label=s[:30] + '...')
axes[1].set_title('Complexity Growth per Generation', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Evolution Generation')
axes[1].set_ylabel('Complexity Score')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 4. Part 3: Code-Specific Data Generation

Code data is among the highest-value training data for LLMs. Three key formats:
1. **Docstring-to-code**: Given a docstring, generate the function body
2. **Code-to-test**: Given a function, generate unit tests
3. **Bug-fix pairs**: Given buggy code, generate the fix

### FAANG Interview Question

**Q: "How would you generate training data for a coding assistant?"**

**A**: Three strategies: (1) Docstring-to-code: extract docstrings from open-source repos, use docstring as prompt and implementation as target. (2) Code-to-test: given a function, generate test cases covering normal, edge, and error paths. (3) Bug injection: take correct code, inject realistic bugs (off-by-one, wrong operator, missing edge case), create (buggy, fixed) pairs. Additionally, use execution-based filtering: only keep pairs where generated code actually passes tests.

In [ ]:
class CodeDataGenerator:
    """Generate training data for coding assistants.

    Produces three types of training pairs:
    - docstring -> code (completion)
    - code -> tests (test generation)
    - buggy code -> fixed code (bug fixing)
    """

    def __init__(self):
        """Initialize with sample Python functions as source material."""
        self.sample_functions = [
            {'name': 'fibonacci', 'doc': 'Return the nth Fibonacci number.',
             'code': 'def fibonacci(n):\n    if n <= 1:\n        return n\n'
                     '    a, b = 0, 1\n    for _ in range(2, n + 1):\n'
                     '        a, b = b, a + b\n    return b'},
            {'name': 'is_palindrome', 'doc': 'Check if a string is a palindrome.',
             'code': 'def is_palindrome(s):\n    s = s.lower().strip()\n'
                     '    return s == s[::-1]'},
            {'name': 'binary_search',
             'doc': 'Find target in sorted list, return index or -1.',
             'code': 'def binary_search(arr, target):\n'
                     '    lo, hi = 0, len(arr) - 1\n    while lo <= hi:\n'
                     '        mid = (lo + hi) // 2\n'
                     '        if arr[mid] == target: return mid\n'
                     '        elif arr[mid] < target: lo = mid + 1\n'
                     '        else: hi = mid - 1\n    return -1'},
            {'name': 'flatten_list', 'doc': 'Flatten a nested list.',
             'code': 'def flatten_list(nested):\n    result = []\n'
                     '    for item in nested:\n'
                     '        if isinstance(item, list):\n'
                     '            result.extend(flatten_list(item))\n'
                     '        else: result.append(item)\n    return result'},
            {'name': 'count_words', 'doc': 'Count word frequencies in a string.',
             'code': 'def count_words(text):\n    words = text.lower().split()\n'
                     '    counts = {}\n    for w in words:\n'
                     '        counts[w] = counts.get(w, 0) + 1\n'
                     '    return counts'},
            {'name': 'max_subarray', 'doc': 'Maximum subarray sum (Kadane).',
             'code': 'def max_subarray(arr):\n    max_sum = current = arr[0]\n'
                     '    for x in arr[1:]:\n'
                     '        current = max(x, current + x)\n'
                     '        max_sum = max(max_sum, current)\n'
                     '    return max_sum'},
            {'name': 'merge_sorted', 'doc': 'Merge two sorted lists.',
             'code': 'def merge_sorted(a, b):\n    result, i, j = [], 0, 0\n'
                     '    while i < len(a) and j < len(b):\n'
                     '        if a[i] <= b[j]: result.append(a[i]); i += 1\n'
                     '        else: result.append(b[j]); j += 1\n'
                     '    result.extend(a[i:]); result.extend(b[j:])\n'
                     '    return result'},
            {'name': 'gcd', 'doc': 'GCD via Euclid algorithm.',
             'code': 'def gcd(a, b):\n    while b:\n'
                     '        a, b = b, a % b\n    return a'},
            {'name': 'matrix_multiply', 'doc': 'Multiply two 2D matrices.',
             'code': 'def matrix_multiply(a, b):\n'
                     '    rows_a, cols_a = len(a), len(a[0])\n'
                     '    cols_b = len(b[0])\n'
                     '    result = [[0]*cols_b for _ in range(rows_a)]\n'
                     '    for i in range(rows_a):\n'
                     '        for j in range(cols_b):\n'
                     '            for k in range(cols_a):\n'
                     '                result[i][j] += a[i][k]*b[k][j]\n'
                     '    return result'},
            {'name': 'is_valid_email', 'doc': 'Check valid email format.',
             'code': 'def is_valid_email(email):\n    import re\n'
                     '    pat = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$"\n'
                     '    return bool(re.match(pat, email))'},
        ]

    def generate_docstring_to_code(self, func: Dict) -> Dict:
        """Extract docstring as prompt, code as target."""
        return {'prompt': f"Write a Python function: {func['doc']}",
                'target': func['code'], 'type': 'docstring_to_code'}

    def generate_code_to_test(self, func: Dict) -> Dict:
        """Generate test template for a given function."""
        test_code = (f"def test_{func['name']}_basic():\n"
                     f"    # Test basic functionality\n"
                     f"    assert {func['name']}  # add assertion\n\n"
                     f"def test_{func['name']}_edge():\n"
                     f"    # Test edge case\n"
                     f"    assert {func['name']}  # add assertion")
        return {'prompt': f"Write unit tests for:\n{func['code']}",
                'target': test_code, 'type': 'code_to_test'}

    def generate_bug_fix_pairs(self, func: Dict) -> List[Dict]:
        """Inject realistic bugs: off-by-one, wrong operator,
        missing edge case, wrong variable."""
        pairs, code = [], func['code']
        if '+ 1' in code:
            buggy = code.replace('+ 1', '+ 2', 1)
            pairs.append({'prompt': f'Fix the bug:\n{buggy}', 'target': code,
                          'bug_type': 'off_by_one', 'type': 'bug_fix'})
        if '<=' in code:
            buggy = code.replace('<=', '<', 1)
            pairs.append({'prompt': f'Fix the bug:\n{buggy}', 'target': code,
                          'bug_type': 'wrong_operator', 'type': 'bug_fix'})
        lines = code.split('\n')
        if len(lines) > 3 and 'if' in lines[1]:
            buggy = lines[0] + '\n' + '\n'.join(lines[3:])
            pairs.append({'prompt': f'Fix the bug:\n{buggy}', 'target': code,
                          'bug_type': 'missing_edge_case', 'type': 'bug_fix'})
        if 'result' in code:
            buggy = code.replace('result', 'res', 1)
            pairs.append({'prompt': f'Fix the bug:\n{buggy}', 'target': code,
                          'bug_type': 'wrong_variable', 'type': 'bug_fix'})
        return pairs


# Generate all data types
code_gen = CodeDataGenerator()
all_code_data = []
for func in code_gen.sample_functions:
    all_code_data.append(code_gen.generate_docstring_to_code(func))
    all_code_data.append(code_gen.generate_code_to_test(func))
    all_code_data.extend(code_gen.generate_bug_fix_pairs(func))

type_counts = collections.Counter(d['type'] for d in all_code_data)
print(f'Total code training pairs: {len(all_code_data)}')
for dtype, count in type_counts.items():
    print(f'  {dtype}: {count}')

ex = [d for d in all_code_data if d['type'] == 'docstring_to_code'][0]
print(f'\nExample docstring-to-code:')
print(f'  Prompt: {ex["prompt"]}')
print(f'  Target: {ex["target"][:80]}...')

bug_ex = [d for d in all_code_data if d['type'] == 'bug_fix']
if bug_ex:
    print(f'\nExample bug-fix ({bug_ex[0]["bug_type"]}):')
    print(f'  Prompt: {bug_ex[0]["prompt"][:100]}...')

---

## 5. Part 4: Quality Filtering

Raw synthetic data is noisy. Quality filtering is critical before training.

### FAANG Interview Question

**Q: "How do you filter low-quality synthetic data?"**

**A**: Multi-stage pipeline: (1) **Length filtering** -- remove too-short or too-long samples. (2) **Perplexity filtering** -- remove samples a language model finds trivially predictable (repetitive) or extremely surprising (gibberish). (3) **Exact deduplication** -- hash-based removal. (4) **Near-deduplication** -- MinHash for fuzzy matching at scale. (5) **N-gram overlap** -- remove samples overlapping with benchmark test sets (decontamination). Typical data loss: 30-60% of raw synthetic data.

In [ ]:
class QualityFilterPipeline:
    """Multi-stage quality filtering for synthetic data.

    Implements length, perplexity, exact dedup, MinHash dedup,
    and n-gram overlap filters.
    """

    def __init__(self):
        """Initialize filter statistics tracker."""
        self.filter_stats: Dict[str, int] = {}

    def _get_text(self, d: Dict) -> str:
        """Extract text content from a data sample."""
        return d.get('response', d.get('target', ''))

    def filter_by_length(
        self, data: List[Dict], min_len: int = 20, max_len: int = 2000
    ) -> List[Dict]:
        """Remove samples outside acceptable length range."""
        before = len(data)
        result = [d for d in data if min_len <= len(self._get_text(d)) <= max_len]
        self.filter_stats['length'] = before - len(result)
        return result

    def filter_by_perplexity(
        self, data: List[Dict],
        threshold_low: float = 1.5, threshold_high: float = 50.0
    ) -> List[Dict]:
        """Filter by perplexity using bigram entropy as proxy.

        Very low entropy = repetitive; very high = gibberish.
        """
        before = len(data)
        result = []
        for d in data:
            words = self._get_text(d).lower().split()
            if len(words) < 3:
                continue
            bigrams = [f'{words[i]} {words[i+1]}' for i in range(len(words) - 1)]
            counts = collections.Counter(bigrams)
            total = len(bigrams)
            entropy = sum(
                -((c / total) * np.log2(c / total)) for c in counts.values()
            )
            ppl = 2.0 ** entropy
            if threshold_low <= ppl <= threshold_high:
                result.append(d)
        self.filter_stats['perplexity'] = before - len(result)
        return result

    def deduplicate_exact(self, data: List[Dict]) -> List[Dict]:
        """Remove exact duplicates using SHA-256 hashing."""
        before = len(data)
        seen: Set[str] = set()
        result = []
        for d in data:
            h = hashlib.sha256(self._get_text(d).encode()).hexdigest()
            if h not in seen:
                seen.add(h)
                result.append(d)
        self.filter_stats['exact_dedup'] = before - len(result)
        return result

    def _minhash_signature(
        self, text: str, n_hashes: int = 100, ngram_size: int = 3
    ) -> List[int]:
        """Compute MinHash signature for a text FROM SCRATCH.

        Algorithm:
        1. Convert text to set of character n-grams (shingles)
        2. Hash each shingle to an integer
        3. For each of n_hashes hash functions h(x) = (a*x + b) mod p,
           compute minimum hash value across all shingles
        4. Return signature vector of minima

        Two documents with high Jaccard similarity will have
        similar MinHash signatures (MinHash theorem).
        """
        text_lower = text.lower()
        shingles = set(
            text_lower[i:i + ngram_size]
            for i in range(len(text_lower) - ngram_size + 1)
        )
        if not shingles:
            return [0] * n_hashes
        # Convert shingles to integer hashes
        shingle_ints = [
            int(hashlib.md5(s.encode()).hexdigest(), 16) for s in shingles
        ]
        # Hash function coefficients: h(x) = (a*x + b) mod p
        large_prime = (1 << 61) - 1
        rng = np.random.RandomState(42)
        a_coeffs = rng.randint(1, large_prime, size=n_hashes)
        b_coeffs = rng.randint(0, large_prime, size=n_hashes)
        signature = []
        for idx in range(n_hashes):
            min_val = float('inf')
            a_i, b_i = int(a_coeffs[idx]), int(b_coeffs[idx])
            for sh in shingle_ints:
                hv = (a_i * sh + b_i) % large_prime
                if hv < min_val:
                    min_val = hv
            signature.append(min_val)
        return signature

    def _minhash_similarity(self, sig1: List[int], sig2: List[int]) -> float:
        """Estimate Jaccard similarity from MinHash signatures."""
        return sum(1 for a, b in zip(sig1, sig2) if a == b) / len(sig1)

    def deduplicate_minhash(
        self, data: List[Dict], threshold: float = 0.8
    ) -> List[Dict]:
        """Near-duplicate removal using MinHash signatures."""
        before = len(data)
        sigs = [self._minhash_signature(self._get_text(d), n_hashes=50)
                for d in data]
        keep = []
        for i in range(len(data)):
            is_dup = any(
                self._minhash_similarity(sigs[i], sigs[j]) >= threshold
                for j in keep
            )
            if not is_dup:
                keep.append(i)
        result = [data[i] for i in keep]
        self.filter_stats['minhash_dedup'] = before - len(result)
        return result

    def filter_ngram_overlap(
        self, data: List[Dict], reference: str,
        n: int = 13, threshold: float = 0.7
    ) -> List[Dict]:
        """Remove samples with high n-gram overlap to reference."""
        before = len(data)
        ref_words = reference.lower().split()
        ref_ngrams = set(
            tuple(ref_words[i:i + n]) for i in range(len(ref_words) - n + 1)
        )
        result = []
        for d in data:
            words = self._get_text(d).lower().split()
            if len(words) < n:
                result.append(d)
                continue
            sample_ng = set(
                tuple(words[i:i + n]) for i in range(len(words) - n + 1)
            )
            overlap = len(sample_ng & ref_ngrams) / max(len(sample_ng), 1)
            if overlap < threshold:
                result.append(d)
        self.filter_stats['ngram_overlap'] = before - len(result)
        return result


# Demonstrate the full filter pipeline
qf_pipeline = QualityFilterPipeline()

# Build raw data with duplicates and edge cases for testing
raw_data = list(generated_data)
raw_data.extend(raw_data[:10])  # add duplicates
raw_data.append({'instruction': 'Hi', 'response': 'Ok'})
raw_data.append({'instruction': 'Test', 'response': 'x ' * 3000})

print(f'Raw data: {len(raw_data)} samples')
stage_counts = [len(raw_data)]
stage_names = ['Raw']

filtered = qf_pipeline.filter_by_length(raw_data, min_len=20, max_len=2000)
stage_counts.append(len(filtered))
stage_names.append('Length')
print(f'After length filter: {len(filtered)}')

filtered = qf_pipeline.filter_by_perplexity(filtered)
stage_counts.append(len(filtered))
stage_names.append('Perplexity')
print(f'After perplexity filter: {len(filtered)}')

filtered = qf_pipeline.deduplicate_exact(filtered)
stage_counts.append(len(filtered))
stage_names.append('Exact Dedup')
print(f'After exact dedup: {len(filtered)}')

filtered = qf_pipeline.deduplicate_minhash(filtered, threshold=0.8)
stage_counts.append(len(filtered))
stage_names.append('MinHash Dedup')
print(f'After MinHash dedup: {len(filtered)}')

print(f'\nFilter statistics: {qf_pipeline.filter_stats}')

# Funnel chart
fig, ax = plt.subplots(figsize=(10, 6))
colors_funnel = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f']
bars = ax.barh(range(len(stage_counts)), stage_counts,
               color=colors_funnel[:len(stage_counts)],
               edgecolor='black', height=0.6)
ax.set_yticks(range(len(stage_names)))
ax.set_yticklabels(stage_names, fontsize=12)
ax.set_xlabel('Number of Samples', fontsize=12)
ax.set_title('Quality Filter Funnel', fontsize=14, fontweight='bold')
ax.invert_yaxis()
for bar, count in zip(bars, stage_counts):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            str(count), va='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
class Decontaminator:
    """Decontamination pipeline for removing benchmark-overlapping data.

    Builds an n-gram index from benchmark test sets and checks
    training samples for overlap. Cross-references Tutorial 43.
    Standard approach: 13-grams (following GPT-3/Llama papers).
    """

    def __init__(self, n: int = 13):
        """Initialize with n-gram size."""
        self.n = n
        self.benchmark_index: Dict[str, Set[tuple]] = {}

    def build_ngram_index(self, benchmark_name: str, texts: List[str]):
        """Build set of n-grams from a benchmark test set."""
        ngram_set: Set[tuple] = set()
        for text in texts:
            words = text.lower().split()
            for i in range(len(words) - self.n + 1):
                ngram_set.add(tuple(words[i:i + self.n]))
        self.benchmark_index[benchmark_name] = ngram_set

    def check_contamination(self, sample: str) -> Dict[str, float]:
        """Check overlap ratio of sample against each benchmark."""
        words = sample.lower().split()
        if len(words) < self.n:
            return {name: 0.0 for name in self.benchmark_index}
        sample_ngrams = set(
            tuple(words[i:i + self.n]) for i in range(len(words) - self.n + 1)
        )
        results = {}
        for name, bench_ngrams in self.benchmark_index.items():
            overlap = len(sample_ngrams & bench_ngrams)
            results[name] = overlap / max(len(sample_ngrams), 1)
        return results

    def decontaminate(
        self, dataset: List[Dict], threshold: float = 0.5
    ) -> Tuple[List[Dict], List[Dict]]:
        """Remove contaminated samples. Returns (clean, contaminated)."""
        clean, contaminated = [], []
        for d in dataset:
            text = d.get('response', d.get('target', ''))
            overlaps = self.check_contamination(text)
            max_overlap = max(overlaps.values()) if overlaps else 0.0
            if max_overlap < threshold:
                clean.append(d)
            else:
                contaminated.append({**d, 'overlap_scores': overlaps})
        return clean, contaminated


# Demo decontamination (5-grams for short demo; 13-grams in production)
decon = Decontaminator(n=5)

benchmark_mmlu = [
    'the system uses approach which provides efficient handling of data',
    'through a well-designed architecture that ensures scalability and reliability',
    'in production environments by implementing proper patterns for error recovery',
]
benchmark_gsm8k = [
    'calculate the total cost of items in the shopping cart after discount',
    'if the train travels at sixty miles per hour for three hours',
]

decon.build_ngram_index('mmlu', benchmark_mmlu)
decon.build_ngram_index('gsm8k', benchmark_gsm8k)

print('Benchmark index sizes:')
for name, ngrams in decon.benchmark_index.items():
    print(f'  {name}: {len(ngrams)} {decon.n}-grams')

clean, contaminated = decon.decontaminate(filtered, threshold=0.3)
print(f'\nDecontamination results (Tutorial 43 cross-reference):')
print(f'  Clean samples: {len(clean)}')
print(f'  Contaminated: {len(contaminated)}')
if contaminated:
    print(f'  Example overlap: {contaminated[0].get("overlap_scores", {})}')

---

## 6. Part 5: Rejection Sampling

Rejection sampling generates K candidate responses per prompt, scores them, and keeps only the best. Used extensively in Llama 2.

### FAANG Interview Question

**Q: "Explain rejection sampling for data generation."**

**A**: For each prompt, generate K responses (typically K=8-64) using temperature sampling. Score each with a reward model or verifier. Keep only the top-N (typically N=1-2). The tradeoff: higher K = better quality but more compute. Llama 2 used rejection sampling to generate training data for their chat model.

In [ ]:
class RejectionSampler:
    """Rejection sampling for high-quality data generation.

    Generates K candidate responses per prompt, scores them,
    and keeps only the top-scoring ones.
    """

    def __init__(self):
        """Initialize scoring vocabulary."""
        self.quality_words = {
            'because', 'therefore', 'however', 'specifically',
            'for example', 'in contrast', 'additionally',
            'furthermore', 'consequently', 'implementation',
        }

    def sample(self, prompt: str, k: int = 16) -> List[str]:
        """Generate K candidate responses.

        TODO: In production, call an LLM with temperature > 0.
        """
        base_words = [
            'the', 'approach', 'involves', 'using', 'a', 'method',
            'that', 'handles', 'the', 'problem', 'by', 'first',
            'analyzing', 'then', 'implementing', 'because', 'this',
            'ensures', 'correctness', 'however', 'there', 'are',
            'tradeoffs', 'specifically', 'in', 'terms', 'of',
            'performance', 'additionally', 'we', 'consider',
            'edge', 'cases', 'for', 'example', 'when', 'input',
            'is', 'empty', 'consequently', 'implementation',
            'must', 'therefore', 'furthermore', 'robust', 'error',
        ]
        responses = []
        for _ in range(k):
            length = random.randint(30, 150)
            responses.append(
                ' '.join(random.choice(base_words) for _ in range(length))
            )
        return responses

    def score(self, response: str) -> float:
        """Score response: length, quality words, diversity."""
        words = response.lower().split()
        score = 0.0
        if 50 <= len(words) <= 100:
            score += 3.0
        elif 30 <= len(words) <= 150:
            score += 1.5
        for qw in self.quality_words:
            if qw in response.lower():
                score += 0.5
        score += (len(set(words)) / max(len(words), 1)) * 3.0
        return score

    def score_and_filter(
        self, responses: List[str], threshold: float = 3.0
    ) -> List[Tuple[str, float]]:
        """Score all responses and filter by threshold."""
        scored = sorted(
            [(r, self.score(r)) for r in responses],
            key=lambda x: x[1], reverse=True,
        )
        return [(r, s) for r, s in scored if s >= threshold]

    def generate_dataset(
        self, prompts: List[str], k: int = 8, keep_top_n: int = 2
    ) -> Tuple[List[Dict], float]:
        """For each prompt, generate K responses, keep top N."""
        dataset = []
        total_gen, total_kept = 0, 0
        for prompt in prompts:
            responses = self.sample(prompt, k=k)
            total_gen += len(responses)
            scored = sorted(
                [(r, self.score(r)) for r in responses],
                key=lambda x: x[1], reverse=True,
            )
            for resp, sc in scored[:keep_top_n]:
                dataset.append({'prompt': prompt, 'response': resp, 'score': sc})
                total_kept += 1
        return dataset, total_kept / max(total_gen, 1)


# Demonstrate rejection sampling
sampler = RejectionSampler()
test_prompts = [
    'Explain binary search.', 'What is a hash table?',
    'Describe the observer pattern.', 'How does TCP work?',
    'What is recursion?', 'Explain polymorphism.',
    'Describe MVC architecture.', 'What is memoization?',
    'How does HTTPS work?', 'Explain database indexing.',
]

rs_dataset, acceptance_rate = sampler.generate_dataset(
    test_prompts, k=16, keep_top_n=2
)
print(f'Rejection Sampling Results:')
print(f'  Prompts: {len(test_prompts)}, Generated/prompt: 16, Kept/prompt: 2')
print(f'  Total dataset: {len(rs_dataset)}')
print(f'  Acceptance rate: {acceptance_rate:.1%}')

# Quality distribution visualization
all_scores = []
kept_scores = [d['score'] for d in rs_dataset]
for p in test_prompts:
    all_scores.extend([sampler.score(r) for r in sampler.sample(p, k=16)])

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(all_scores, bins=20, alpha=0.5, color='gray',
        label='All generated', edgecolor='black')
ax.hist(kept_scores, bins=20, alpha=0.7, color='green',
        label='Kept (top-2)', edgecolor='black')
ax.set_title('Rejection Sampling: Quality Distribution',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Quality Score')
ax.set_ylabel('Count')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 7. Part 6: Preference Pair Generation

For DPO/RLHF training (Tutorial 41), we need (prompt, chosen, rejected) triples. Rejection sampling naturally produces ranked responses for preference pairs.

### FAANG Interview Question

**Q: "How do you generate preference pairs from ranked completions?"**

**A**: Two strategies: (1) **All-pairs**: Given K ranked responses, create C(K,2) pairs where the higher-ranked is chosen. Maximizes data but includes small-margin pairs (noisy). (2) **Best-vs-worst**: Only pair top against bottom. Fewer pairs but cleaner signal. In practice, best-vs-worst with margin filtering works well.

In [ ]:
class PreferencePairGenerator:
    """Generate preference pairs from ranked completions.

    Converts ranked responses into (chosen, rejected) pairs
    suitable for DPO training (Tutorial 41 format).
    """

    def generate_pairs_from_ranks(
        self, prompt: str,
        ranked_responses: List[Tuple[str, float]],
        strategy: str = 'best_vs_worst',
    ) -> List[Dict]:
        """Create preference pairs from ranked responses.

        Args:
            prompt: The instruction prompt.
            ranked_responses: (response, score) sorted desc.
            strategy: 'all_pairs' or 'best_vs_worst'.
        """
        pairs = []
        if strategy == 'all_pairs':
            for i in range(len(ranked_responses)):
                for j in range(i + 1, len(ranked_responses)):
                    c_r, c_s = ranked_responses[i]
                    r_r, r_s = ranked_responses[j]
                    pairs.append({
                        'prompt': prompt, 'chosen': c_r,
                        'rejected': r_r, 'margin': c_s - r_s,
                    })
        elif strategy == 'best_vs_worst' and len(ranked_responses) >= 2:
            best_r, best_s = ranked_responses[0]
            worst_r, worst_s = ranked_responses[-1]
            pairs.append({
                'prompt': prompt, 'chosen': best_r,
                'rejected': worst_r, 'margin': best_s - worst_s,
            })
        return pairs

    def generate_pairs_bestofn(
        self, prompt: str, responses: List[str],
        scorer, n_pairs: int = 3,
    ) -> List[Dict]:
        """Best-of-N pairing: best response vs random non-best."""
        scored = sorted(
            [(r, scorer(r)) for r in responses],
            key=lambda x: x[1], reverse=True,
        )
        best_r, best_s = scored[0]
        pairs = []
        for rej_r, rej_s in scored[1:min(n_pairs + 1, len(scored))]:
            pairs.append({
                'prompt': prompt, 'chosen': best_r,
                'rejected': rej_r, 'margin': best_s - rej_s,
            })
        return pairs


# Generate preference dataset
pref_gen = PreferencePairGenerator()
preference_dataset = []
for prompt in test_prompts:
    responses = sampler.sample(prompt, k=8)
    scored = sorted(
        [(r, sampler.score(r)) for r in responses],
        key=lambda x: x[1], reverse=True,
    )
    pairs = pref_gen.generate_pairs_from_ranks(
        prompt, scored[:4], strategy='all_pairs'
    )
    preference_dataset.extend(pairs)

margins = [p['margin'] for p in preference_dataset]
print(f'Preference Dataset (Tutorial 41 format):')
print(f'  Total pairs: {len(preference_dataset)}')
print(f'  Mean margin: {np.mean(margins):.3f}')
print(f'  Std margin: {np.std(margins):.3f}')
print(f'  Min/Max margin: {np.min(margins):.3f} / {np.max(margins):.3f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(margins, bins=25, color='mediumpurple', edgecolor='black', alpha=0.8)
ax.axvline(x=np.mean(margins), color='red', linestyle='--',
           linewidth=2, label=f'Mean={np.mean(margins):.2f}')
ax.set_title('Preference Margin Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Score Margin (chosen - rejected)')
ax.set_ylabel('Count')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 8. Part 7: Data Mixing and Curriculum

Modern LLMs are trained on carefully mixed data from multiple sources with curriculum scheduling.

### FAANG Interview Question

**Q: "How do you decide data mixing ratios?"**

**A**: Start with proportional mixing, then tune based on downstream performance. Key principles: (1) Over-represent high-quality sources (code, math). (2) Under-represent web crawl despite volume. (3) Use per-domain validation loss to monitor balance. (4) Curriculum: start diverse and easy, gradually increase difficulty. Llama 2 used ~90% pre-training, ~10% fine-tuning data, with fine-tuning heavily over-represented.

In [ ]:
class DataMixer:
    """Weighted data mixing from multiple sources."""

    def create_mixture(
        self, datasets: Dict[str, List], ratios: Dict[str, float],
        total_samples: int = 100,
    ) -> List[Dict]:
        """Create mixed dataset via weighted sampling."""
        total_weight = sum(ratios.values())
        normalized = {k: v / total_weight for k, v in ratios.items()}
        mixture = []
        for source, ratio in normalized.items():
            n = int(total_samples * ratio)
            sampled = [random.choice(datasets[source]) for _ in range(n)]
            for s in sampled:
                mixture.append({**s, 'source': source})
        random.shuffle(mixture)
        return mixture

    def curriculum_schedule(
        self, dataset: List[Dict], difficulty_fn,
        n_epochs: int = 3,
    ) -> List[List[Dict]]:
        """Create easy-to-hard curriculum ordering.

        Each epoch includes data up to the current difficulty level.
        """
        scored = sorted(
            [(d, difficulty_fn(d)) for d in dataset],
            key=lambda x: x[1],
        )
        chunk_size = len(scored) // n_epochs
        epochs = []
        for i in range(n_epochs):
            end = min((i + 1) * chunk_size, len(scored))
            epoch_data = [s[0] for s in scored[:end]]
            random.shuffle(epoch_data)
            epochs.append(epoch_data)
        return epochs


class DataVersioner:
    """Track dataset versions with hashes, metadata, lineage."""

    def __init__(self):
        """Initialize version registry."""
        self.versions: List[Dict] = []

    def register_version(
        self, name: str, data: List[Dict],
        parent_version: Optional[str] = None,
        description: str = '',
    ) -> Dict:
        """Register a new dataset version with content hash."""
        content_str = json.dumps([str(d) for d in data], sort_keys=True)
        content_hash = hashlib.sha256(content_str.encode()).hexdigest()[:16]
        version = {
            'name': name, 'hash': content_hash,
            'n_samples': len(data), 'parent': parent_version,
            'description': description,
        }
        self.versions.append(version)
        return version

    def get_lineage(self, name: str) -> List[str]:
        """Get the full lineage chain for a version."""
        version_map = {v['name']: v for v in self.versions}
        lineage, current = [], name
        while current and current in version_map:
            lineage.append(current)
            current = version_map[current].get('parent')
        return lineage


# Demo: Mix 3 datasets
mixer = DataMixer()
code_data = [{'text': f'code sample {i}', 'type': 'code'} for i in range(30)]
math_data = [{'text': f'math problem {i}', 'type': 'math'} for i in range(20)]
general_data = [{'text': f'general text {i}', 'type': 'general'} for i in range(50)]

datasets = {'code': code_data, 'math': math_data, 'general': general_data}
ratios = {'code': 0.4, 'math': 0.3, 'general': 0.3}

mixed = mixer.create_mixture(datasets, ratios, total_samples=100)
source_counts = collections.Counter(d['source'] for d in mixed)

print('Data Mixing Results:')
print(f'  Total samples: {len(mixed)}')
for src, cnt in source_counts.items():
    print(f'  {src}: {cnt} ({cnt/len(mixed):.0%})')

# Curriculum demo
def difficulty_fn(sample):
    """Difficulty proxy based on text length."""
    return len(sample.get('text', ''))

epochs = mixer.curriculum_schedule(mixed, difficulty_fn, n_epochs=3)
print(f'\nCurriculum Schedule:')
for i, epoch in enumerate(epochs):
    avg_d = np.mean([difficulty_fn(d) for d in epoch])
    print(f'  Epoch {i+1}: {len(epoch)} samples, avg difficulty={avg_d:.1f}')

# Version tracking
versioner = DataVersioner()
versioner.register_version('raw_v1', code_data, description='Raw code data')
versioner.register_version('filtered_v1', code_data[:20],
                           parent_version='raw_v1', description='After filtering')
versioner.register_version('mixed_v1', mixed,
                           parent_version='filtered_v1', description='Final mix')
print(f'\nData Versioning:')
for v in versioner.versions:
    print(f'  {v["name"]}: {v["n_samples"]} samples, '
          f'hash={v["hash"]}, parent={v["parent"]}')
print(f'  Lineage: {versioner.get_lineage("mixed_v1")}')

---

## 9. FAANG Interview Questions

### Q1: "Design an end-to-end synthetic data pipeline for training a chat model."

**A**: Five stages: (1) **Seed collection**: Curate 500-1000 high-quality instruction-response pairs across target domains. (2) **Self-Instruct + Evol-Instruct**: Generate 50K-100K instructions with controlled complexity distribution. (3) **Response generation**: For each instruction, generate K=16 responses with rejection sampling, keep top-2. (4) **Quality filtering**: Length filter, perplexity filter, MinHash dedup, decontamination against held-out benchmarks. (5) **Preference pair generation**: From the K ranked responses, create (chosen, rejected) pairs for DPO (Tutorial 41). Final output: ~30K instruction pairs + ~50K preference pairs.

---

### Q2: "When should you use synthetic data vs human-annotated data?"

**A**: Synthetic excels at: scaling volume (10x-100x cheaper), coverage (generate rare domains), consistency (no annotator disagreement). Human excels at: capturing nuance, edge cases models miss, establishing ground truth, creative tasks. Best practice: human data for seed sets and evaluation, synthetic for scaling volume. Always validate synthetic quality with human assessment on a sample.

---

### Q3: "How do you maintain quality at scale when generating millions of synthetic samples?"

**A**: (1) Cascade filtering: cheap filters first (length, regex), expensive last (LLM-based scoring). (2) Execution-based validation for code: only keep samples where generated code passes tests. (3) Consistency checks: generate multiple responses, check agreement. (4) Periodic human audit: sample 1%, rate quality to calibrate filters. (5) Diversity monitoring: track n-gram diversity, topic distribution, difficulty distribution.

---

### Q4: "Why is decontamination important? What happens if you skip it?"

**A**: Without decontamination, training data may contain copies of benchmark test samples. This inflates scores without genuine capability improvement -- memorization, not reasoning. Detection: 13-gram overlap between training data and benchmark test sets. Impact: GPT-3 showed 1-2% score inflation on contaminated benchmarks. See Tutorial 43.

---

### Q5: "How do you generate alignment data without human annotators?"

**A**: Three approaches: (1) **Constitutional AI** (Anthropic): model critiques and revises its own responses according to principles. (2) **Rejection sampling + reward model**: generate K responses, score with RM, pair best vs worst. (3) **LLM-as-judge**: stronger model ranks weaker model's responses. All scale to millions of pairs without humans, but should be validated periodically. See Tutorial 41 for DPO training on these pairs.

In [ ]:
# Comprehensive 4-Panel Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Data source distribution (pie chart)
src_labels = list(source_counts.keys())
src_values = list(source_counts.values())
colors_pie = ['#4e79a7', '#f28e2b', '#e15759']
axes[0, 0].pie(
    src_values, labels=src_labels, autopct='%1.0f%%',
    colors=colors_pie, startangle=90, textprops={'fontsize': 11}
)
axes[0, 0].set_title('Data Source Distribution',
                      fontsize=13, fontweight='bold')

# Panel 2: Quality score distribution per source
np.random.seed(42)
q_code = np.random.normal(7.5, 1.2, 40)
q_math = np.random.normal(6.8, 1.5, 30)
q_general = np.random.normal(5.5, 2.0, 30)
bp = axes[0, 1].boxplot(
    [q_code, q_math, q_general],
    labels=['Code', 'Math', 'General'],
    patch_artist=True,
)
for patch, color in zip(bp['boxes'], colors_pie):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[0, 1].set_title('Quality Score by Source',
                      fontsize=13, fontweight='bold')
axes[0, 1].set_ylabel('Quality Score')
axes[0, 1].grid(True, alpha=0.3)

# Panel 3: Token length distribution
np.random.seed(42)
token_lengths = np.concatenate([
    np.random.lognormal(4.0, 0.6, 60),
    np.random.lognormal(4.5, 0.4, 40),
])
axes[1, 0].hist(token_lengths, bins=30, color='steelblue',
                edgecolor='black', alpha=0.7)
axes[1, 0].axvline(x=np.median(token_lengths), color='red',
                    linestyle='--', linewidth=2,
                    label=f'Median={np.median(token_lengths):.0f}')
axes[1, 0].set_title('Token Length Distribution',
                      fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Token Length')
axes[1, 0].set_ylabel('Count')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Panel 4: Filter funnel (bar chart)
funnel_stages = ['Raw', 'Length', 'Perplexity', 'Exact Dedup', 'MinHash']
funnel_vals = stage_counts[:5] if len(stage_counts) >= 5 else stage_counts
colors_bar = ['#4e79a7', '#59a14f', '#f28e2b', '#e15759', '#76b7b2']
bars = axes[1, 1].bar(
    funnel_stages[:len(funnel_vals)], funnel_vals,
    color=colors_bar[:len(funnel_vals)], edgecolor='black',
)
for bar, val in zip(bars, funnel_vals):
    axes[1, 1].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
        str(val), ha='center', va='bottom', fontweight='bold', fontsize=11,
    )
axes[1, 1].set_title('Filter Pipeline Funnel',
                      fontsize=13, fontweight='bold')
axes[1, 1].set_ylabel('Number of Samples')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Synthetic Data Generation Pipeline Overview',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Pipeline summary:')
print(f'  Self-Instruct: {len(generated_data)} pairs')
print(f'  Code data: {len(all_code_data)} pairs')
print(f'  Rejection sampling: {len(rs_dataset)} pairs')
print(f'  Preference pairs: {len(preference_dataset)} pairs')
print(f'  Final mixed dataset: {len(mixed)} samples')

---

## 10. Key Takeaways

### Foundation (Know These Cold)
- [ ] Self-Instruct: seed -> generate -> filter -> expand pool
- [ ] Evol-Instruct: depth (harder) + breadth (different domain) evolution
- [ ] Rejection sampling: generate K, score, keep top-N
- [ ] MinHash deduplication: shingles -> hash functions -> signature -> similarity
- [ ] Decontamination: 13-gram overlap against benchmark test sets (Tutorial 43)

### Implementation (Be Able to Code)
- [ ] `SelfInstructPipeline.run_pipeline()` -- full generation loop with ROUGE-L filtering
- [ ] `EvolInstructPipeline.evolve()` -- chained depth/breadth evolution
- [ ] `QualityFilterPipeline._minhash_signature()` -- from-scratch MinHash
- [ ] `Decontaminator.decontaminate()` -- n-gram index build and check
- [ ] `PreferencePairGenerator.generate_pairs_from_ranks()` -- ranked to (chosen, rejected) for Tutorial 41

### Production Patterns (Interview Differentiator)
- [ ] Quality > quantity: 10K high-quality samples beats 1M noisy ones
- [ ] Multi-stage filtering: cheap filters first, expensive last
- [ ] Data versioning: hash, metadata, lineage for every dataset version
- [ ] Curriculum learning: easy-to-hard ordering improves convergence
- [ ] Decontamination is mandatory before any benchmark evaluation

### Common Mistakes (Avoid These)
- [ ] Training on synthetic data without quality filtering
- [ ] Ignoring decontamination (inflated benchmark scores)
- [ ] Using the same model for generation and scoring (circular)
- [ ] Not tracking data lineage (irreproducible results)
- [ ] Over-relying on a single data source without mixing

---

**Next**: Tutorial 42 -- Coding Agents from Scratch